In [12]:
import os
import json
import logging
import torch
import pandas as pd
import re

from collections import Counter
from pathlib import Path
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

import psycopg2
from psycopg2 import sql, extras
import logging
import csv
import os
import json


In [13]:
# Set up basic logging to see success/failure messages
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [14]:
DB_PARAMS = {
    'dbname': 'thecall',
    'user': 'postgres',        
    'password': 'password',    
    'host': 'localhost',           
    'port': '5432'                 
}

In [15]:
# ── Device setup ──────────────────────────────────────────────────────────────
# Priority: CUDA (Nvidia) > XPU (Intel Arc/Data Center) > CPU
if torch.cuda.is_available():
    DEVICE = 0          # HuggingFace pipeline uses integer device index for CUDA
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif hasattr(torch, 'xpu') and torch.xpu.is_available():
    # Intel XPU — pipeline doesn't accept 'xpu' string directly,
    # so we move the model manually after loading (see model cell below).
    DEVICE = 'xpu'
    DEVICE_NAME = 'Intel XPU'
else:
    DEVICE = -1         # CPU fallback
    DEVICE_NAME = 'CPU'

print(f'PyTorch version : {torch.__version__}')
print(f'Selected device : {DEVICE_NAME}')

PyTorch version : 2.7.0+xpu
Selected device : Intel XPU


In [16]:
# ── Configuration ─────────────────────────────────────────────────────────────

MODEL_NAME  = 'Jean-Baptiste/roberta-large-ner-english'
BATCH_SIZE  = 32

# Aggregation strategy: 'simple' merges sub-word tokens back into words.
# 'first' and 'max' are alternatives if you see odd splits.
AGGREGATION = 'simple'

In [17]:
# ── Model loading ─────────────────────────────────────────────────────────────
logging.info(f'Loading model: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model     = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
model.eval()

if DEVICE == 'xpu':
    # Intel extension must be imported before moving the model
    import intel_extension_for_pytorch as ipex
    model = model.to('xpu')
    # Wrap with IPEX for optimised inference
    model = ipex.optimize(model, dtype=torch.float32)
    # HuggingFace pipeline needs a plain device string when passing a pre-loaded model
    pipeline_device = 'xpu'
    print(f'Intel PyTorch Extension version: {ipex.__version__}')
else:
    pipeline_device = DEVICE   # int (0 for CUDA) or -1 (CPU)

ner_pipe = pipeline(
    'ner',
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy=AGGREGATION,
    device=pipeline_device,
)

logging.info('Pipeline ready.')

INFO: Loading model: Jean-Baptiste/roberta-large-ner-english
2026-04-08 23:21:26,414 - _logger.py - IPEX - INFO - Currently split master weight for xpu only support sgd
2026-04-08 23:21:26,421 - _logger.py - IPEX - INFO - Conv BatchNorm folding failed during the optimize process.
2026-04-08 23:21:26,427 - _logger.py - IPEX - INFO - Linear BatchNorm folding failed during the optimize process.
Device set to use xpu
INFO: Pipeline ready.


Intel PyTorch Extension version: 2.7.10+xpu


In [18]:
def connect_to_db(params):
    """Establishes a connection to the PostgreSQL database."""
    conn = None
    try:
        logging.info("Attempting to connect to PostgreSQL database...")
        conn = psycopg2.connect(**params)
        # Set autocommit to False so we can control transactions (needed for UPDATE)
        conn.autocommit = False 
        logging.info("Connection successful.")
        return conn
    except psycopg2.Error as e:
        logging.error(f"Error connecting to the database: {e}")
        return None

In [19]:
def collect_data_from_db(conn, table_schema, table_name):
    full_table_name = f"{table_schema}.{table_name}"
    try:
        with conn.cursor() as cur:
            # Added ORDER BY and LIMIT 100 for a consistent test set
            query = sql.SQL("SELECT id, json_raw FROM {}.{} WHERE json_raw IS NOT NULL ORDER BY id LIMIT 100").format(
                sql.Identifier(table_schema),
                sql.Identifier(table_name)
            )
            cur.execute(query)
            return cur.fetchall()
    except psycopg2.Error as e:
        logging.error(f"Error fetching data: {e}")
        return []
        

In [ ]:
def is_valid_entity(text):
    """Checks if the entity is substantial enough to be kept."""
    # 1. Remove entities that are just punctuation or single dots
    if re.match(r'^[.\s\d\W_]+$', text):
        return False
    
    # 2. Filter out very short noise (e.g., "Mo", "s", "M") 
    # Adjust length if you want to keep 2-letter states like "KS"
    if len(text) < 3 and text.upper() not in ["KS", "MO"]:
        return False
        
    return True

def clean_text(text):
    """Cleans up OCR artifacts and trailing punctuation."""
    # Remove leading/trailing non-word characters (like  or periods)
    text = re.sub(r'^[^\w]+|[^\w]+$', '', text)
    return text.strip()

In [20]:
def ner_task(all_data, ner_pipe):
    """
    all_data: List of tuples (id, json_raw)
    Returns: List of tuples (entity_type, entity_value, fk_filelist)
    """
    results = []
    logging.info(f"Starting NER on {len(all_data)} articles...")

    for row_id, json_raw in tqdm(all_data, desc="Processing Articles"):
        if not json_raw:
            continue
            
        # 1. Extract text from the JSON structure
        full_text_list = []
        for page in json_raw.get('pages', []):
            for frame in page.get('frames', []):
                # Using .get('text') for the frame content
                frame_text = frame.get('text', '').strip()
                if frame_text:
                    full_text_list.append(frame_text)
        
        article_text = " ".join(full_text_list)
        
        if not article_text:
            continue

        try:
            # 2. Run RoBERTa NER Pipeline
            # RoBERTa usually has a 512 token limit (~3000-4000 chars)
            entities = ner_pipe(article_text[:4000]) 
            
            for ent in entities:
                raw_word = ent['word']
                
                # Apply Cleaning
                cleaned_val = clean_entity_text(raw_word)
                
                # Apply Validation
                if is_valid_entity(cleaned_val):
                    # results format: (entity_type, entity_value, fk_filelist)
                    results.append((
                        ent['entity_group'], # Usually PER, ORG, LOC, or MISC
                        cleaned_val,         # The cleaned entity text
                        row_id               # The FK back to your database
                    ))
                    
        except Exception as e:
            logging.error(f"NER failed for ID {row_id}: {e}")

    return results

In [21]:
def update_dbs(conn, table_schema, output_table, results):
    if not results:
        logging.info("No results to insert.")
        return

    try:
        with conn.cursor() as cur:
            # 1. Clear the table before inserting new test results
            truncate_query = sql.SQL("TRUNCATE TABLE {}.{} RESTART IDENTITY CASCADE;").format(
                sql.Identifier(table_schema),
                sql.Identifier(output_table)
            )
            cur.execute(truncate_query)
            logging.info(f"Table {table_schema}.{output_table} truncated.")

            # 2. Insert into the entities table as defined
            insert_query = sql.SQL("""
                INSERT INTO {}.{} (entity_type, entity_value, fk_filelist)
                VALUES %s
            """).format(
                sql.Identifier(table_schema),
                sql.Identifier(output_table)
            )
            
            extras.execute_values(cur, insert_query, results)
            conn.commit()
            logging.info(f"Test complete: Inserted {len(results)} entities.")
            
    except psycopg2.Error as e:
        conn.rollback()
        logging.error(f"Database operation failed: {e}")

In [24]:
def main():
    conn = connect_to_db(DB_PARAMS)
    if conn:
        try:
            table_schema = 'articles'
            table_name = 'filelist'
            output_table = 'entities'

            # 1. Fetch existing entries
            all_data = collect_data_from_db(conn, table_schema, table_name)
            
            if not all_data:
                logging.warning("No data retrieved from DB. Exiting.")
                return

            # 2. Perform NER
            results = ner_task(all_data)
           
            # 3. Batch update the DB
            update_dbs(conn, table_schema, output_table, results)
            
        finally:
            if conn:
                conn.close()
                logging.info("Database connection closed.")

In [25]:
if __name__ == '__main__':
    main()

INFO: Attempting to connect to PostgreSQL database...
INFO: Connection successful.
INFO: Starting NER on 100 articles...
Processing Articles: 100%|███████████████████████████████████████████████████████████| 100/100 [00:04<00:00, 25.00it/s]
INFO: Test complete: Inserted 1324 entities.
INFO: Database connection closed.
